# ENTERPRISE KNOWLEDGE MINING SYSTEM

#### CONFIGURATIONS

In [1]:
from dataclasses import dataclass
from pathlib import Path
import os

BASE_URL = "https://export.arxiv.org/api/query"

@dataclass
class PipelineConfig:
    member_mode: bool = True
    search_query: str = "all:computer"
    start: int = 0
    max_results: int = 100           # Change to 100 or 1000 to scale
    pdf_dir: Path = Path("pdfs")
    min_page_length: int = 100
    chunk_size: int = 800
    chunk_overlap: int = 50       
    batch_size: int = 500           
    embedding_model: str = "text-embedding-3-small"
    rag_model: str = "gpt-4.1-mini"
    chroma_path: str = "./chroma_db"
    collection_name: str = "research_papers"
    download_workers: int = min(12, (os.cpu_count() or 4) * 2)    
    process_workers: int = max(1, (os.cpu_count() or 4) - 1)        
    arxiv_page_size: int = 100     
    request_delay: float = 3.0 

config = PipelineConfig()
config.pdf_dir.mkdir(exist_ok=True)

params = {
    "search_query": config.search_query,
    "start": config.start,
    "max_results": config.max_results,
}

In [2]:
from huggingface_hub import hf_hub_download, HfApi, list_repo_files
from huggingface_hub.utils import disable_progress_bars
import dotenv
import threading

dotenv.load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
HF_REPO_ID = os.getenv("HF_REPO_ID")

_arxiv_semaphore = threading.Semaphore(4)

_hf_files_cache = None
_hf_cache_lock = threading.Lock()

hf_api = HfApi(token=HF_TOKEN)


def get_hf_files():
    global _hf_files_cache
    with _hf_cache_lock:
        if _hf_files_cache is None:
            try:
                _hf_files_cache = set(list_repo_files(
                    repo_id=HF_REPO_ID, 
                    repo_type="dataset",
                    token=HF_TOKEN
                ))
                print(f"HF repo contains {len(_hf_files_cache)} files.")
            except Exception as e:
                print(f"Could not fetch HF file list: {e}")
                _hf_files_cache = set()
        return _hf_files_cache


def pdf_exists_in_hf(filename):
    return filename in get_hf_files()

def upload_to_hf(local_path):
    filename = os.path.basename(local_path)
    disable_progress_bars()
    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=filename,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with _hf_cache_lock:
        if _hf_files_cache is not None:
            _hf_files_cache.add(filename)


def download_from_hf(filename, local_path):
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        repo_type="dataset",
        local_dir=os.path.dirname(local_path),
    )  

c:\Users\jm201\OneDrive\Documents\GitHub\Enterprise-Knowledge-Mining-System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### DOCUMENT INGESTION

In [3]:
import xml.etree.ElementTree as ET
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import pymupdf as fitz

NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom"
}

In [4]:
import urllib.request
import urllib.error

# ==== HTTP Request Helper ====

def make_request(url, delay=5, retries=3):
    for attempt in range(retries):
        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "arxiv-data-project/1.0 (research script)"}
            )
            with urllib.request.urlopen(request) as response:
                data = response.read()
            time.sleep(delay)
            return data
        except urllib.error.HTTPError as e:
            if e.code == 429:
                wait = delay * (2 ** attempt)
                time.sleep(wait)
            else:
                raise
    raise Exception("Max retries exceeded")

In [5]:
# ==== Parsing Helper Functions ====

def get_text(entry, tag):
    return entry.findtext(tag, default="", namespaces=NS).strip()

def extract_authors(entry):
    return [
        author.find("atom:name", NS).text.strip()
        for author in entry.findall("atom:author", NS)
    ]

def extract_categories(entry):
    return [
        cat.attrib.get("term")
        for cat in entry.findall("atom:category", NS)
        if cat.attrib.get("term")
    ]

def extract_pdf_link(entry):
    return next(
        (
            link.attrib.get("href")
            for link in entry.findall("atom:link", NS)
            if link.attrib.get("title") == "pdf"
            and link.attrib.get("type") == "application/pdf"
            and link.attrib.get("rel") == "related"
        ),
        None
    )

def extract_primary_category(entry):
    primary = entry.find("arxiv:primary_category", NS)
    return primary.attrib.get("term") if primary is not None else None

def extract_paper(entry):
    return {
        "arxiv_id": get_text(entry, "atom:id").split("/")[-1],
        "title": get_text(entry, "atom:title"),
        "summary": get_text(entry, "atom:summary"),
        "published": get_text(entry, "atom:published"),
        "authors": extract_authors(entry),
        "primary_category": extract_primary_category(entry),
        "categories": extract_categories(entry),
        "pdf_link": extract_pdf_link(entry),
    }

def parse_response(response):
    root = ET.fromstring(response)
    return [extract_paper(entry) for entry in root.findall("atom:entry", NS)]

In [6]:
import urllib.parse

# ==== NEW: Paginated arxiv Fetcher ====
# CHANGE: Beyond 100 results per page the API becomes unreliable and slow.  
# This function issues multiple requests of `page_size` each, sleeping `delay` 
# seconds between them to respect arxiv's rate-limit policy.

def fetch_all_papers(params, base_url=BASE_URL, page_size=100, delay=3.0):
    total_needed = params.get("max_results", 10)
    all_papers = []
    fetched = 0

    while fetched < total_needed:
        batch = min(page_size, total_needed - fetched)
        page_params = {
            **params,
            "start": params.get("start", 0) + fetched,
            "max_results": batch,
        }
        query_url = base_url + "?" + urllib.parse.urlencode(page_params)
        print(f"Fetching papers {fetched + 1}-{fetched + batch} …")
        response = make_request(query_url, delay=delay)
        page_papers = parse_response(response)
        if not page_papers:
            print("  arxiv returned 0 entries — stopping early.")
            break
        all_papers.extend(page_papers)
        fetched += len(page_papers)

    print(f"Fetched {len(all_papers)} paper records.")
    return all_papers

#### EXTRACTING PDF AND CONVERTING TO MARKDOWN

In [7]:
pdf_dir = str(config.pdf_dir)
os.makedirs(pdf_dir, exist_ok=True)

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", name)

def download_pdf(url, save_path, overwrite=False, delay=1.0):
    if os.path.exists(save_path) and not overwrite:
        return save_path
    data = make_request(url, delay=delay, retries=3)
    with open(save_path, "wb") as f:
        f.write(data)
    return save_path

In [8]:
# ==== NEW: Parallel Download then Parallel Parse ====
# CHANGE: The replacement uses two ThreadPoolExecutors in sequence:
#   1. Download phase  – up to `download_workers` concurrent HTTP downloads.
#   2. Parse phase     – up to `process_workers` concurrent pymupdf4llm calls.

def _download_one(paper):
    if not paper.get("pdf_link"):
        return paper, None
    
    safe_title = sanitize_filename(paper["title"])
    filename = f"{safe_title}.pdf"
    local_path = os.path.join(pdf_dir, filename)

    # 1. Local cache check
    if os.path.exists(local_path):
        return paper, local_path
    
    # 2. Hugging Face hit
    if pdf_exists_in_hf(filename):
        try:
            download_from_hf(filename, local_path)
            return paper, local_path
        except Exception as exc:
            print(f"  HF download failed for '{paper['title']}': {exc}")
            if config.member_mode:
                return paper, None
            
    # 3. Arxiv download
    if config.member_mode:
        print(f"  Not in HF yet, skipping (member mode): {filename}")
        return paper, None
    
    try:
        with _arxiv_semaphore:
            download_pdf(paper["pdf_link"], local_path, delay=1.0)
        upload_to_hf(local_path)
        return paper, local_path
    except Exception as exc:
        print(f"  Download/Upload failed for '{paper['title']}': {exc}")
        return paper, None


def _parse_one(paper_path_pair):
    paper, path = paper_path_pair
    try:
        doc = fitz.open(path)
        pages = []
        for page_num, page in enumerate(doc, start=1):
            pages.append({
                "text": page.get_text("text"),
                "raw_text": page.get_text("text"),
                "metadata": {
                    "page_number": page_num,
                    "page_count": len(doc),
                    "format": doc.metadata.get("format", ""),
                    "creationDate": doc.metadata.get("creationDate", ""),
                }
            })
        doc.close()
        return {**paper, "pages": pages}
    except Exception as exc:
        print(f"  Parse failed for '{paper['title']}': {exc}")
        return None


def process_papers_parallel(papers, download_workers=8, process_workers=4):
    # --- Phase 1: parallel downloads ---
    downloadable = [p for p in papers if p.get("pdf_link")]
    skipped = len(papers) - len(downloadable)
    if skipped:
            print(f"  Skipped {skipped} papers with no PDF link.")

    print(f"Downloading {len(downloadable)} PDFs with {download_workers} workers…")
    downloaded = [] 
    with ThreadPoolExecutor(max_workers=download_workers) as executor:
        futures = {executor.submit(_download_one, paper): paper for paper in downloadable}
        for future in as_completed(futures):
            paper, path = future.result()
            if path:
                downloaded.append((paper, path))
            else:
                print(f"  Download failed for '{paper['title']}'")
    print(f"  {len(downloaded)}/{len(downloadable)} PDFs downloaded.")

    # --- Phase 2: parallel PDF → markdown parsing ---
    print(f"Parsing PDFs with {process_workers} workers…")
    processed_papers = []
    with ThreadPoolExecutor(max_workers=process_workers) as executor:
        futures = {executor.submit(_parse_one, pair): pair for pair in downloaded}
        for future in as_completed(futures):
            result = future.result()
            if result:
                processed_papers.append(result)
    print(f"  {len(processed_papers)} papers parsed successfully.")

    return processed_papers

#### DATA CLEANING FOR CHUNKING

In [9]:
def clean_heading(text: str) -> str:
    text = text.strip()
    text = re.sub(r'[*_`#]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_section_blocks(page_text: str):
    pattern = re.compile(
        r'^(##.*|_?\*{0,2}Abstract\*{0,2}_?.*|_?\*{0,2}Keywords:.*\*{0,2}_?)$',
        re.IGNORECASE | re.MULTILINE
    )
    matches = list(pattern.finditer(page_text))
    section_blocks = []
    for i, match in enumerate(matches):
        section = re.sub(r'[*_`#]+', '', match.group()).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(page_text)
        text = page_text[start:end].strip()
        if text:
            section_blocks.append({"section": section, "text": text})
    return section_blocks


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"(?m)^\s*#{1,6}\s*", " ", text)
    text = re.sub(r"(?m)^\s*[-*_~=`]{2,}\s*$", " ", text)
    text = re.sub(r"[_*`~]+", " ", text)
    text = re.sub(r"(?<!\w)#(?=\w)", "", text)
    text = text.replace("#", " ")
    text = re.sub(r"([A-Za-z])-\s*\n\s*([A-Za-z])", r"\1\2", text)
    text = re.sub(r"(?im)^\s*page\s*\d+(\s*of\s*\d+)?\s*$", " ", text)
    text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)
    text = re.sub(r"[—–]+", " - ", text)
    text = re.sub(r"^[\s\.,;:-]+", "", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_page_text(text: str, page_number=None) -> str:
    text = clean_text(text)
    if not text:
        return ""
    if page_number == 1:
        match = re.search(r"\babstract\b\s*[:-]?\s*", text, flags=re.IGNORECASE)
        if match:
            text = text[match.start():]
    return re.sub(r"\s+", " ", text).strip()


def clean_processed_papers(processed_papers, min_length):
    cleaned_papers = []
    for paper in processed_papers:
        clean_pages = []
        for page in paper["pages"]:
            raw_text = page.get("text", "")
            cleaned = clean_page_text(
                raw_text,
                page_number=page["metadata"].get("page_number")
            )
            if cleaned and len(cleaned) >= min_length:
                clean_pages.append({
                    "text": cleaned,
                    "raw_text": raw_text,
                    "metadata": page["metadata"]
                })
        cleaned_papers.append({
            **{k: v for k, v in paper.items() if k != "pages"},
            "pages": clean_pages
        })
    return cleaned_papers

In [10]:
# ==== Main ingestion entry point ====
# CHANGE: A two-step call that (a) pages through arxiv and (b) uses parallel download + parse.

from debug_utils import preview_cleaned_paper, preview_section_blocks

# Step 1: fetch paper metadata (paginated for 100/1000)
papers = fetch_all_papers(
    params=params,
    base_url=BASE_URL,
    page_size=config.arxiv_page_size,
    delay=config.request_delay,
)

# Step 2: download + parse in parallel
processed_papers = process_papers_parallel(
    papers,
    download_workers=config.download_workers,
    process_workers=config.process_workers,
)

# Step 3: clean
cleaned_papers = clean_processed_papers(processed_papers, config.min_page_length)

preview_cleaned_paper(cleaned_papers)

Fetching papers 1-100 …
Fetched 100 paper records.
  100/100 PDFs downloaded.
Parsing PDFs with 15 workers…
  100 papers parsed successfully.

==== A Secure Multi-Party Computation Protocol for Malicious Computation Prevention for preserving privacy during Data Mining ====

--- Page 0 Preview ---
Metadata: {'page_number': 1, 'page_count': 6, 'format': 'PDF 1.4', 'creationDate': "D:20090807111424+05'00'"}
Text Preview:
 ABSTRACT- Secure Multi-Party Computation (SMC) allows parties with similar background to compute results upon their private data, minimizing the threat of disclosure. The exponential increase in sensitive data that needs to be passed upon networked computers and the stupendous growth of internet has precipitated vast opportunities for cooperative computation, where parties come together to facilitate computations and draw out conclusions that are mutually beneficial; at the same time aspiring to


In [11]:
def build_chunk_paper_metadata(paper):
    categories = paper.get("categories", [])
    return {
        "paper_id": paper.get("arxiv_id", "paper_0"),
        "title": paper.get("title", "untitled"),
        "primary_category": paper.get("primary_category", ""),
        "categories": ", ".join(categories) if isinstance(categories, list) else str(categories),
        "authors": ", ".join(paper.get("authors", [])) if paper.get("authors") else "",
        "published": paper.get("published", "")
    }


def build_chunk_page_metadata(page):
    metadata = page.get("metadata", {})
    return {
        "creation_date": metadata.get("creationDate", ""),
        "page_number": metadata.get("page_number", ""),
        "page_count": metadata.get("page_count", ""),
        "format": metadata.get("format", "")
    }


def remove_known_running_headers(text):
    if not text:
        return ""
    text = re.sub(
        r"(?i)\(?IJCSIS\)?\s+International Journal of Computer Science and Information Security,\s*Vol\.\s*7,\s*No\.\s*1,\s*2010",
        " ", text
    )
    return text


def clean_block_text(text):
    return clean_text(remove_known_running_headers(text))


def should_keep_section(section_name, paper_title="", excluded_sections=None):
    excluded_sections = excluded_sections or set()
    if section_name.strip().title() in excluded_sections:
        return False
    return True


def build_section_block(section, text, paper, page, page_block_index):
    block_text = clean_block_text(text)
    if not block_text:
        return None
    return {
        "section": section,
        "text": block_text,
        "metadata": {
            **build_chunk_paper_metadata(paper),
            **build_chunk_page_metadata(page),
            "page_block_index": page_block_index
        }
    }


def extract_blocks_from_page(page, paper=None, excluded_sections=None):
    paper = paper or {}
    source_text = page.get("raw_text") or page.get("text", "")
    extracted_blocks = extract_section_blocks(source_text)
    blocks = []
    for block_index, block in enumerate(extracted_blocks):
        section = block.get("section", "").strip() or "Unknown"
        if not should_keep_section(section, paper.get("title", ""), excluded_sections):
            continue
        section_block = build_section_block(
            section=section,
            text=block.get("text", ""),
            paper=paper,
            page=page,
            page_block_index=len(blocks)
        )
        if section_block:
            blocks.append(section_block)
    return blocks


def append_continuation_to_previous_block(previous_block, page):
    continuation_text = clean_block_text(page.get("text") or page.get("raw_text", ""))
    if continuation_text:
        previous_block["text"] = f"{previous_block['text']} {continuation_text}".strip()
    return previous_block


def extract_blocks_from_paper(paper, excluded_sections=None):
    paper_blocks = []
    for page in paper.get("pages", []):
        page_blocks = extract_blocks_from_page(page, paper=paper, excluded_sections=excluded_sections)
        if page_blocks:
            paper_blocks.extend(page_blocks)
        elif paper_blocks:
            append_continuation_to_previous_block(paper_blocks[-1], page)
        else:
            fallback_block = build_section_block(
                section="Unknown",
                text=page.get("text") or page.get("raw_text", ""),
                paper=paper,
                page=page,
                page_block_index=0
            )
            if fallback_block:
                paper_blocks.append(fallback_block)
    return paper_blocks


def extract_blocks_from_papers(cleaned_papers, excluded_sections=None):
    all_blocks = []
    for paper in cleaned_papers:
        all_blocks.extend(extract_blocks_from_paper(paper, excluded_sections=excluded_sections))
    return all_blocks

In [12]:
EXCLUDED_SECTIONS = {
    "References",
    "Acknowledgment",
    "Acknowledgements"
}

def select_papers_by_title(cleaned_papers, title_contains):
    title_contains = title_contains.lower()
    return [
        paper for paper in cleaned_papers
        if title_contains in paper.get("title", "").lower()
    ]

# To debug a single paper uncomment and set the title:
# SECTION_PAPER_TITLE = "Vision Based Game Development"
# section_papers = select_papers_by_title(cleaned_papers, SECTION_PAPER_TITLE)

section_papers = cleaned_papers
all_blocks = extract_blocks_from_papers(section_papers, excluded_sections=EXCLUDED_SECTIONS)
preview_section_blocks(all_blocks)


PAGE: 1
SECTION: ABSTRACT- Secure Multi-Party Computation (SMC) allows
TEXT SAMPLE: parties with similar background to compute results upon their private data, minimizing the threat of disclosure. The exponential increase in sensitive data that needs to be passed upon networked computers and the stupendous growth of internet has precipitated vast opportunities for cooperative computation, where parties come together to facilitate computations and draw out conclusions that are mutually beneficial; at the same time aspiring to keep their private data secure. These computations ar

PAGE: 1
SECTION: Abstract
TEXT SAMPLE: The extremum graph is a succinct representation of the Morse decomposition of a scalar ﬁeld. It has increasingly become a useful data structure that supports topological feature directed visualization of 2D / 3D scalar ﬁelds, and enables dimensionality reduction together with exploratory analysis of high dimensional scalar ﬁelds. Current methods that employ the extremum g

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def build_text_splitter(chunk_size=500, chunk_overlap=200):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""]
    )


def chunk_section_blocks(blocks, text_splitter=None, starting_chunk_index=0):
    text_splitter = text_splitter or build_text_splitter()
    chunks = []
    global_chunk_index = starting_chunk_index
    for block in blocks:
        block_text = block.get("text", "")
        if not block_text.strip():
            continue
        splits = text_splitter.split_text(block_text)
        for page_chunk_index, split_text in enumerate(splits):
            split_text = re.sub(r"^[\s\.,;:-]+", "", split_text).strip()
            if split_text:
                metadata = {
                    **block.get("metadata", {}),
                    "section": block.get("section", "Unknown"),
                    "page_chunk_index": page_chunk_index,
                    "chunk_index": global_chunk_index
                }
                chunks.append({"text": split_text, "metadata": metadata})
                global_chunk_index += 1
    return chunks


def chunk_cleaned_papers(cleaned_papers, text_splitter=None, excluded_sections=None):
    blocks = extract_blocks_from_papers(cleaned_papers, excluded_sections=excluded_sections)
    return chunk_section_blocks(blocks, text_splitter=text_splitter)


text_splitter = build_text_splitter(
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap
)
chunks = chunk_section_blocks(all_blocks, text_splitter=text_splitter)

print("Total chunks:", len(chunks))
print(chunks[0] if chunks else "No chunks created")

Total chunks: 9696
{'text': 'parties with similar background to compute results upon their private data, minimizing the threat of disclosure. The exponential increase in sensitive data that needs to be passed upon networked computers and the stupendous growth of internet has precipitated vast opportunities for cooperative computation, where parties come together to facilitate computations and draw out conclusions that are mutually beneficial; at the same time aspiring to keep their private data secure. These computations are generally required to be done between competitors, who are obviously weary of eachothers intentions', 'metadata': {'paper_id': '0908.0994v1', 'title': 'A Secure Multi-Party Computation Protocol for Malicious Computation Prevention for preserving privacy during Data Mining', 'primary_category': 'cs.CR', 'categories': 'cs.CR, cs.DB', 'authors': 'Dr. Durgesh Kumar Mishra, Neha Koria, Nikhil Kapoor, Ravish Bahety', 'published': '2009-08-07T07:22:06Z', 'creation_date': 

In [14]:
def text_for_ner(chunk):
    text = (chunk.get("text") or "")
    md = chunk.get("metadata", {})
    text = re.sub(r"[*_`~^—–]+", " ", text)
    text = re.sub(r"^[\s\.,;:-]+", "", text)
    if md.get("page_number") == 1:
        title = (md.get("title") or "")
        if title:
            text = re.sub(re.escape(title), " ", text, flags=re.IGNORECASE)
        authors = md.get("authors", "")
        author_list = (
            [a.strip() for a in authors.split(",") if a.strip()]
            if isinstance(authors, str)
            else [str(a).strip() for a in authors if str(a).strip()]
        )
        for author in author_list:
            text = re.sub(rf"\b{re.escape(author)}\b", " ", text, flags=re.IGNORECASE)
        parts = re.split(r"\babstract\b[:\-\s]*", text, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) == 2:
            text = parts[1]
    return re.sub(r"\s+", " ", text).strip()[:400]

#### Entity Extraction

In [15]:
import subprocess, json
import spacy


# CHANGE: switched to en_core_web_sm for 100–1000 paper scale.
# CHANGE: added n_process for multi-core NER.

nlp = spacy.load("en_core_web_sm", enable=["ner"])  # switch to en_core_web_md for higher accuracy

In [16]:
def clean_entities(raw_entities):
    cleaned = []
    seen = set()
    for ent_text, ent_label in raw_entities:
        ent_text = ent_text.strip()
        if not ent_text:
            continue
        if len(ent_text) < 3:
            continue
        if re.fullmatch(r"\d+(\.\d+)?", ent_text):
            continue
        if ent_label in {"CARDINAL", "ORDINAL"}:
            continue
        if re.fullmatch(r"[^\w]+", ent_text):
            continue
        key = (ent_text.lower(), ent_label)
        if key not in seen:
            seen.add(key)
            cleaned.append((ent_text, ent_label))
    return cleaned


# def enrich_chunks_with_entities(chunks, nlp_model=None, batch_size=512, n_process=1):
#     # CHANGE: n_process parameter wired through so callers can enable multicore.
#     nlp_model = nlp_model or nlp
#     ner_texts = [text_for_ner(chunk) if len(chunk.get("text", "")) > 80 else "" for chunk in chunks]
#     docs = nlp_model.pipe(ner_texts, batch_size=batch_size, n_process=n_process)
#     for chunk, doc, ner_text in zip(chunks, docs, ner_texts):
#         raw_entities = [(ent.text, ent.label_) for ent in doc.ents]
#         cleaned = clean_entities(raw_entities)
#         chunk["metadata"]["entities"] = ", ".join(text for text, _ in cleaned)
#         chunk["metadata"]["entity_labels"] = ", ".join(label for _, label in cleaned)
#         chunk["metadata"]["ner_text"] = ner_text
#     return chunks

def enrich_chunks_with_entities(chunks):
    ner_texts = [text_for_ner(chunk) if len(chunk.get("text", "")) > 80 else "" for chunk in chunks]

    proc = subprocess.run(
        ["python", "ner_worker.py"],
        input=json.dumps(ner_texts),
        text=True,
        capture_output=True
    )
    all_entities = json.loads(proc.stdout)

    for chunk, raw_entities, ner_text in zip(chunks, all_entities, ner_texts):
        cleaned = clean_entities(raw_entities)
        chunk["metadata"]["entities"] = ", ".join(text for text, _ in cleaned)
        chunk["metadata"]["entity_labels"] = ", ".join(label for _, label in cleaned)
        chunk["metadata"]["ner_text"] = ner_text
    return chunks


chunks = enrich_chunks_with_entities(chunks)

print(chunks[0]["metadata"])

{'paper_id': '0908.0994v1', 'title': 'A Secure Multi-Party Computation Protocol for Malicious Computation Prevention for preserving privacy during Data Mining', 'primary_category': 'cs.CR', 'categories': 'cs.CR, cs.DB', 'authors': 'Dr. Durgesh Kumar Mishra, Neha Koria, Nikhil Kapoor, Ravish Bahety', 'published': '2009-08-07T07:22:06Z', 'creation_date': "D:20090807111424+05'00'", 'page_number': 1, 'page_count': 6, 'format': 'PDF 1.4', 'page_block_index': 0, 'section': 'ABSTRACT- Secure Multi-Party Computation (SMC) allows', 'page_chunk_index': 0, 'chunk_index': 0, 'entities': '', 'entity_labels': '', 'ner_text': 'parties with similar background to compute results upon their private data, minimizing the threat of disclosure. The exponential increase in sensitive data that needs to be passed upon networked computers and the stupendous growth of internet has precipitated vast opportunities for cooperative computation, where parties come together to facilitate computations and draw out conc

In [17]:
METADATA_KEYS = [
    "paper_id", "title", "section", "primary_category", "categories",
    "authors", "published", "creation_date", "page_number", "page_count",
    "format", "page_block_index", "page_chunk_index", "chunk_index",
    "entities", "entity_labels"
]

def build_final_chunks(chunks):
    final_chunks = []
    for i, chunk in enumerate(chunks):
        metadata = {key: chunk["metadata"].get(key, "") for key in METADATA_KEYS}
        if not metadata["paper_id"]:
            metadata["paper_id"] = "paper_0"
        if metadata["chunk_index"] == "":
            metadata["chunk_index"] = i
        final_chunks.append({"text": chunk["text"], "metadata": metadata})
    return final_chunks


final_chunks = build_final_chunks(chunks)

for i, chunk in enumerate(final_chunks[:5]):
    print("\nFinal Chunk", i, "metadata:", chunk["metadata"])
    print("Final Chunk", i, "text preview:", chunk["text"][:180])


Final Chunk 0 metadata: {'paper_id': '0908.0994v1', 'title': 'A Secure Multi-Party Computation Protocol for Malicious Computation Prevention for preserving privacy during Data Mining', 'section': 'ABSTRACT- Secure Multi-Party Computation (SMC) allows', 'primary_category': 'cs.CR', 'categories': 'cs.CR, cs.DB', 'authors': 'Dr. Durgesh Kumar Mishra, Neha Koria, Nikhil Kapoor, Ravish Bahety', 'published': '2009-08-07T07:22:06Z', 'creation_date': "D:20090807111424+05'00'", 'page_number': 1, 'page_count': 6, 'format': 'PDF 1.4', 'page_block_index': 0, 'page_chunk_index': 0, 'chunk_index': 0, 'entities': '', 'entity_labels': ''}
Final Chunk 0 text preview: parties with similar background to compute results upon their private data, minimizing the threat of disclosure. The exponential increase in sensitive data that needs to be passed 

Final Chunk 1 metadata: {'paper_id': '0908.0994v1', 'title': 'A Secure Multi-Party Computation Protocol for Malicious Computation Prevention for preserving pr

#### EMBEDDING WITH OPENAI AND STORING IN CHROMADB

In [18]:
from openai import OpenAI, RateLimitError
import chromadb

dotenv.load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_KEY"))

chroma_client = chromadb.PersistentClient(path=config.chroma_path)

try:
    chroma_client.delete_collection(config.collection_name)
    print(f"Deleted existing collection: {config.collection_name}")
except:
    print(f"No existing collection named: {config.collection_name}")

collection = chroma_client.get_or_create_collection(
    name=config.collection_name,
    configuration={"hnsw": {"space": "cosine"}}
)
print("Chroma collection ready.")

Deleted existing collection: research_papers
Chroma collection ready.


In [19]:
# ==== Embedding + Storage ====
# CHANGE: embed_chunks now uses concurrent.futures.ThreadPoolExecutor to fire
# multiple embedding API calls in parallel instead of one-at-a-time.
MAX_EMBED_WORKERS = 6

def get_embeddings(texts, model="text-embedding-3-small"):
    response = client.embeddings.create(model=model, input=texts)
    return [item.embedding for item in response.data]


def _embed_batch(args):
    """Submit one batch to OpenAI; returns (batch_index, embeddings)."""
    batch_index, batch_docs, model = args
    max_retries = 6
    for attempt in range(max_retries):
        try:
            embeddings = get_embeddings(batch_docs, model=model)
            return batch_index, embeddings
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"Rate limit hit for batch {batch_index + 1}, retrying in {wait} seconds…")
            time.sleep(wait)


def embed_chunks(final_chunks, config, max_workers=MAX_EMBED_WORKERS):
    documents = [chunk["text"] for chunk in final_chunks]
    metadatas = [chunk["metadata"] for chunk in final_chunks]
    ids = [
        f"{chunk['metadata']['paper_id']}_chunk_{chunk['metadata']['chunk_index']}"
        for chunk in final_chunks
    ]

    # Build batch argument list
    batches = [
        (i // config.batch_size, documents[i:i + config.batch_size], config.embedding_model)
        for i in range(0, len(documents), config.batch_size)
    ]

    # Pre-allocate results list so order is preserved regardless of completion order
    all_embeddings_ordered = [None] * len(batches)

    print(f"Embedding {len(documents)} chunks in {len(batches)} batches ({max_workers} concurrent)…")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_embed_batch, batch): batch[0] for batch in batches}
        for future in as_completed(futures):
            batch_index, embeddings = future.result()
            all_embeddings_ordered[batch_index] = embeddings
            print(f"  Batch {batch_index + 1}/{len(batches)} done.")

    # Flatten in order
    all_embeddings = [emb for batch in all_embeddings_ordered for emb in batch]

    print("\n--- Embedding Stats ---")
    print("Total embeddings created:", len(all_embeddings))
    print("Embedding dimension:", len(all_embeddings[0]) if all_embeddings else 0)

    return documents, metadatas, ids, all_embeddings


def store_chunks_in_chroma(collection, documents, metadatas, ids, embeddings):
    # CHANGE: added batched upsert for large collections.
    # ChromaDB's .add() with 50k+ items can exceed memory limits. Batching keeps
    # RAM flat and gives visible progress.
    CHROMA_UPSERT_BATCH = 2000
    total = len(documents)
    for i in range(0, total, CHROMA_UPSERT_BATCH):
        collection.add(
            ids=ids[i:i + CHROMA_UPSERT_BATCH],
            documents=documents[i:i + CHROMA_UPSERT_BATCH],
            metadatas=metadatas[i:i + CHROMA_UPSERT_BATCH],
            embeddings=embeddings[i:i + CHROMA_UPSERT_BATCH],
        )
        print(f"  Stored chunks {i + 1}–{min(i + CHROMA_UPSERT_BATCH, total)}/{total}")
    print(f"Stored {total} chunks in ChromaDB.")
    return total


documents, metadatas, ids, embeddings = embed_chunks(final_chunks, config)
stored_count = store_chunks_in_chroma(collection, documents, metadatas, ids, embeddings)
print("\nStored count:", stored_count)

Embedding 9696 chunks in 20 batches (6 concurrent)…
  Batch 1/20 done.
  Batch 3/20 done.
  Batch 6/20 done.
  Batch 5/20 done.
  Batch 2/20 done.
  Batch 4/20 done.
  Batch 7/20 done.
  Batch 10/20 done.
  Batch 9/20 done.
  Batch 8/20 done.
  Batch 11/20 done.
  Batch 14/20 done.
  Batch 15/20 done.
Rate limit hit for batch 12, retrying in 1 seconds…
Rate limit hit for batch 13, retrying in 1 seconds…
Rate limit hit for batch 17, retrying in 1 seconds…
Rate limit hit for batch 16, retrying in 1 seconds…
Rate limit hit for batch 19, retrying in 1 seconds…
  Batch 18/20 done.
  Batch 13/20 done.
Rate limit hit for batch 12, retrying in 2 seconds…
Rate limit hit for batch 17, retrying in 2 seconds…
Rate limit hit for batch 16, retrying in 2 seconds…
Rate limit hit for batch 20, retrying in 1 seconds…
  Batch 20/20 done.
Rate limit hit for batch 19, retrying in 2 seconds…
  Batch 19/20 done.
  Batch 12/20 done.
Rate limit hit for batch 16, retrying in 4 seconds…
Rate limit hit for batch 

### Testing Retrieval

In [20]:
def retrieve_chunks(query, n_results=3, filter_category=None):
    query_embedding = get_embeddings([query], model=config.embedding_model)[0]
    where = {"primary_category": filter_category} if filter_category else None
    return collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where
    )


test_queries = [
    "What is the main purpose of the proposed HCI system?",
    "What is Hough transform and how is it used in this paper?",
    "How does blink detection work in the eye ROI?"
]

for query in test_queries:
    print("\n" + "="*50)
    print(f"\nSearch results for query: '{query}'")
    results = retrieve_chunks(query, n_results=3)
    for i in range(len(results["documents"][0])):
        print(f"\nResult {i+1}")
        print("ID:", results["ids"][0][i])
        print("Metadata:", results["metadatas"][0][i])
        print("Text:", results["documents"][0][i][:400])
    print("\n" + "="*50)



Search results for query: 'What is the main purpose of the proposed HCI system?'

Result 1
ID: 1002.2191v1_chunk_1981
Metadata: {'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'format': 'PDF 1.4', 'page_number': 1, 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'entity_labels': 'ORG, PERSON, PERSON', 'chunk_index': 1981, 'published': '2010-02-10T19:46:07Z', 'page_count': 7, 'section': 'Abstract— A Human Computer Interface (HCI)', 'primary_category': 'cs.HC', 'page_block_index': 0, 'entities': 'SSR Filter, Hough, HCI', 'categories': 'cs.HC, cs.CV, cs.MM', 'page_chunk_index': 1, 'creation_date': "D:20100202231332+05'00'"}
Text: Keywords: Human Computer Interface (HCI), SSR Filter, Hough transform I. INTRODUCTION One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to in

### SEMANTIC SEARCH ENGINE

In [21]:
def normalize_entity_set(entities: str):
    return {e.strip().lower() for e in entities.split(",") if e.strip()}


def compute_entity_overlap(entities: str, query: str):
    entity_set = normalize_entity_set(entities)
    query_terms = {term.lower().strip() for term in query.split() if term.strip()}
    return len(entity_set.intersection(query_terms))


def compute_hybrid_score(cosine_similarity, entity_overlap, use_hybrid=False, entity_bonus=0.5):
    if cosine_similarity is None:
        return None
    if not use_hybrid:
        return cosine_similarity
    return cosine_similarity + (entity_overlap * entity_bonus)


def semantic_search(query, n_results=5, min_similarity=0.25, filter_category=None, use_hybrid=False, entity_bonus=0.5):
    results = retrieve_chunks(query, n_results=n_results, filter_category=filter_category)
    formatted_results = []
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    ids = results.get("ids", [[]])[0]
    distances = results.get("distances", [[]])[0] if "distances" in results else []
    for i, doc_text in enumerate(documents):
        metadata = metadatas[i] if i < len(metadatas) else {}
        chunk_id = ids[i] if i < len(ids) else f"chunk_{i}"
        cosine_distance = distances[i] if i < len(distances) else None
        cosine_similarity = 1 - cosine_distance if cosine_distance is not None else None
        if cosine_similarity is not None and cosine_similarity < min_similarity:
            continue
        primary_category = metadata.get("primary_category", "")
        entities = metadata.get("entities", "")
        print(f"\nEntities for chunk {chunk_id}: {entities}")
        entity_overlap = compute_entity_overlap(entities, query)
        hybrid_score = compute_hybrid_score(cosine_similarity, entity_overlap, use_hybrid=use_hybrid, entity_bonus=entity_bonus)
        formatted_results.append({
            "rank": i + 1,
            "chunk_id": chunk_id,
            "cosine_distance": cosine_distance,
            "cosine_similarity": cosine_similarity,
            "hybrid_score": hybrid_score,
            "entity_overlap": entity_overlap,
            "text": doc_text,
            "paper_id": metadata.get("paper_id", ""),
            "title": metadata.get("title", ""),
            "section": metadata.get("section", ""),
            "primary_category": primary_category,
            "categories": metadata.get("categories", ""),
            "authors": metadata.get("authors", ""),
            "published": metadata.get("published", ""),
            "page_number": metadata.get("page_number", ""),
            "page_block_index": metadata.get("page_block_index", ""),
            "page_chunk_index": metadata.get("page_chunk_index", ""),
            "chunk_index": metadata.get("chunk_index", ""),
            "entities": entities,
            "entity_labels": metadata.get("entity_labels", "")
        })
    if use_hybrid:
        formatted_results = sorted(
            formatted_results,
            key=lambda x: x["hybrid_score"] if x["hybrid_score"] is not None else -1,
            reverse=True
        )
        for idx, result in enumerate(formatted_results, start=1):
            result["rank"] = idx
    return formatted_results


from debug_utils import display_search_results


def semantic_search_engine(query, n_results=5):
    results = semantic_search(query, n_results=n_results)
    return {"query": query, "total_results": len(results), "results": results}


search_output = semantic_search_engine("What is the main purpose of the proposed HCI system?", n_results=3)
print("Query:", search_output["query"])
print("Total results:", search_output["total_results"])
display_search_results(search_output["results"])


Entities for chunk 1002.2191v1_chunk_1981: SSR Filter, Hough, HCI

Entities for chunk 1510.05963v1_chunk_5359: 

Entities for chunk 1002.2191v1_chunk_1986: 
Query: What is the main purpose of the proposed HCI system?
Total results: 3
Rank: 1
Paper ID: 1002.2191v1
Title: Vision Based Game Development Using Human Computer Interaction
Section: Abstract— A Human Computer Interface (HCI)
Primary Category: cs.HC
Categories: cs.HC, cs.CV, cs.MM
Authors: S. Sumathi, S. K. Srivatsa, M. Uma Maheswari
Published: 2010-02-10T19:46:07Z
Page: 1 | Block: 0 | Page Chunk: 1 | Global Chunk: 1981
Cosine Similarity (approx): 0.5884997248649597
Entities: SSR Filter, Hough, HCI
Entity Labels: ORG, PERSON, PERSON
Preview: Keywords: Human Computer Interface (HCI), SSR Filter, Hough transform I. INTRODUCTION One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features

### RAG PIPELINE IMPLEMENTATION

In [22]:
RAG_SYSTEM_PROMPT = """
You are an academic research assistant for an enterprise knowledge mining system built on arXiv papers.

Instructions:
- Answer questions only from the retrieved context.
- Treat the retrieved text as excerpts from research papers.
- Prefer precise academic wording.
- Summarize contributions, methods, datasets, results, or limitations only if they are supported by the context.
- If the answer is incomplete or missing from the context, say that explicitly.
- Do not include source tags or citation brackets such as [Source 1] in the answer text.
- Write only the answer and ensure it is formatted concisely and clearly.
"""


def generate_answer(query, context, max_tokens=200, temperature=0.2):
    response = client.chat.completions.create(
        model=config.rag_model,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Use the retrieved research-paper context below to answer the question.\n\n"
                    f"Context:\n{context}\n\n"
                    f"Question: {query}\n\n"
                    "Answer:"
                )
            }
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )
    return response.choices[0].message.content.strip()


def rag_query(query, n_results=3, min_similarity=0.25, filter_category=None, use_hybrid=False, entity_bonus=0.5):
    search_results = semantic_search(
        query=query, n_results=n_results, min_similarity=min_similarity,
        filter_category=filter_category, use_hybrid=use_hybrid, entity_bonus=entity_bonus
    )
    if not search_results:
        return {
            "query": query,
            "answer": "I cannot answer this question based on the provided context. No relevant context was found.",
            "sources": []
        }
    context_parts = []
    sources = []
    for i, result in enumerate(search_results):
        context_parts.append(
            f"[Source {i+1}: {result['title']}, "
            f"Section: {result['section']}, "
            f"Page {result['page_number']}, "
            f"Chunk {result['chunk_index']}]\n"
            f"{result['text']}"
        )
        sources.append({
            "source_number": i + 1,
            "chunk_id": result["chunk_id"],
            "paper_id": result["paper_id"],
            "title": result["title"],
            "section": result["section"],
            "primary_category": result["primary_category"],
            "categories": result["categories"],
            "page_number": result["page_number"],
            "chunk_index": result["chunk_index"],
            "cosine_similarity": result["cosine_similarity"],
            "hybrid_score": result["hybrid_score"],
            "entity_overlap": result["entity_overlap"]
        })
    context = "\n\n".join(context_parts)
    answer = generate_answer(query, context)
    return {"query": query, "answer": answer, "sources": sources}


queries = [
    "What is HCI and how is it used in the system?"
]

for q in queries:
    print("\n" + "="*50)
    print(f"\nRAG response for query: '{q}'")
    response = rag_query(q, n_results=3)
    print("Answer:\n", response["answer"])
    print("\nSources:")
    for s in response["sources"]:
        print(s)



RAG response for query: 'What is HCI and how is it used in the system?'

Entities for chunk 1002.2191v1_chunk_1981: SSR Filter, Hough, HCI

Entities for chunk 1510.05963v1_chunk_5359: 

Entities for chunk 2506.10523v2_chunk_3413: HPC, HP2C, Information Technologies/Opera
Answer:
 Human-Computer Interface (HCI) is the point of communication between a human and a computer, aiming to use human features to interact with the computer. In the described system, HCI tracks the user's movements using a video camera, specifically capturing and monitoring the tip of the user's nose with a webcam. These tracked movements are then translated into corresponding movements of the mouse pointer on the screen, enabling interaction events that communicate with the computer.

Sources:
{'source_number': 1, 'chunk_id': '1002.2191v1_chunk_1981', 'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'Abstract— A Human Computer Interface (HCI)', 'pri

### EXPERIMENTAL: RERANKING CHUNKS BASED ON ENTITY OVERLAP

In [23]:
entity_queries = [
    ("What is HCI and how is it used in the system?", {"use_hybrid": True}),
]

for q, opts in entity_queries:
    print("\n" + "=" * 50)
    print(f"\nRAG response for query: '{q}'")
    response = rag_query(q, n_results=3, **opts)
    print("Answer:\n", response["answer"])
    print("\nSources:")
    for s in response["sources"]:
        print(s)



RAG response for query: 'What is HCI and how is it used in the system?'

Entities for chunk 1002.2191v1_chunk_1981: SSR Filter, Hough, HCI

Entities for chunk 1510.05963v1_chunk_5359: 

Entities for chunk 2506.10523v2_chunk_3413: HPC, HP2C, Information Technologies/Opera
Answer:
 Human-Computer Interface (HCI) is the point of communication between a human and a computer, aiming to use human features to interact with the computer. In the described system, HCI is implemented by tracking the user's movements with a video camera, specifically capturing and monitoring the tip of the user's nose via a webcam. These tracked movements are then translated into corresponding movements of the mouse pointer on the screen, enabling interaction with the computer through natural human motion.

Sources:
{'source_number': 1, 'chunk_id': '1002.2191v1_chunk_1981', 'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'Abstract— A Human Computer

### EXPERIMENTAL: FILTER CATEGORY

This will be tested when we scale up documents once the categories vary among documents.

This test explores whether NER-derived metadata could provide additional retrieval benefit (other than enriching metadata) through hybrid re-ranking. `use_hybrid=True` re-ranks chunks that have higher entity overlap after retrieving semantically as usual. However, there is currently no significant improvement in re-ranking the chunks based on entity overlap. The general semantic search flow using `retrieve_chunks` already performs well without this enhancement.